In [2]:
import torch

from src.plot import show_single_sample, show_grid
from src.evaluation import FrechetInceptionDistance
from src.data_import import load_fashion_mnist, load_olivetti

from baseline_models import VAE, DiffusionModel

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [26]:
_, test_fm = load_fashion_mnist(batch_size = 32, class_subset=[9])
_, test_oliv = load_olivetti(batch_size = 32)

def loader_to_tensor(loader, n_samples, device="cpu"):
    images = []
    collected = 0

    for batch_images, _ in loader:
        remaining = n_samples - collected

        # Prende solo le immagini necessarie dall'ultimo batch
        batch_images = batch_images[:remaining]

        images.append(batch_images)
        collected += len(batch_images)

        if collected >= n_samples:
            break

    return torch.cat(images, dim=0).to(device)


In [ ]:
Base_VAE_fm = VAE(latent_dim=64)
Base_VAE_olivetti = VAE(img_size=64, latent_dim=64)

Base_Diff_fm = DiffusionModel()
Base_Diff_olivetti = DiffusionModel(img_size=64)

Base_Diff_fm.unet.load_state_dict(torch.load("models/FMNIST/DIFFUSION/diffusion_fashionmnist_class9.pt"))
Base_VAE_fm.load_state_dict(torch.load("models/FMNIST/VAE/vae_fashionmnist_class9.pt"))

Base_Diff_olivetti.unet.load_state_dict(torch.load("models/OLIVETTI/DIFFUSION/diffusion_olivetti.pt"))
Base_VAE_olivetti.load_state_dict(torch.load("models/OLIVETTI/VAE/vae_olivetti.pt"))

<All keys matched successfully>

In [8]:
vae_oliv_samp = Base_VAE_olivetti.sample(n=100, device=DEVICE)
diff_oliv_samp = Base_Diff_olivetti.sample(n=100, device=DEVICE)

vae_fm_samp = Base_VAE_fm.sample(n=100, device=DEVICE)
diff_fm_samp = Base_Diff_fm.sample(n=100, device=DEVICE)

In [ ]:
show_single_sample(vae_fm_samp, cmap="gray")
show_single_sample(diff_fm_samp, cmap="gray")
show_single_sample(vae_oliv_samp, cmap="gray")
show_single_sample(diff_oliv_samp, cmap="gray")

In [28]:
test_samp_fm = loader_to_tensor(test_fm, n_samples=200)
test_samp_oliv = loader_to_tensor(test_oliv, n_samples=200)

In [32]:
from src.evaluation import evaluate_sampling_fid


print("FASHION MNIST - VAE")
evaluate_sampling_fid(real_images=test_samp_fm, fake_images=vae_fm_samp)

print("OLIVETTI - VAE")
evaluate_sampling_fid(real_images=test_samp_oliv, fake_images=vae_oliv_samp)

print("FASHION MNIST - DIFFUSION")
evaluate_sampling_fid(real_images=test_samp_fm, fake_images=diff_fm_samp)

print("OLIVETTI - DIFFUSION")
evaluate_sampling_fid(real_images=test_samp_oliv, fake_images=diff_oliv_samp)

FASHION MNIST - VAE
FID: 186.2517547607422
OLIVETTI - VAE
FID: 430.4732971191406
FASHION MNIST - DIFFUSION
FID: 111.22093200683594
OLIVETTI - DIFFUSION
FID: 420.92828369140625


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from src.data_import import load_fashion_mnist, load_olivetti
from src.model import EBM, EnergyHead, EBM_Old
from src.plot import visualize_heads, visualize_head_gradients, visualize_head_abs_gradients, show_grid, show_single_sample
from src.sampler import ReplaySampler

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Data
TRAIN_CLASSES = [9]
BS = 32
IMG_SHAPE = (1, 28, 28)
IMG_SHAPE_Olivetti = (1, 64, 64)
# Model
N_HEADS = 3

# Sampler
BUFFER_SIZE = 8
NOISE_FRACTION = 0.05
NOISE_STD = 0.005

model_fm = EBM(image_shape=IMG_SHAPE, n_heads=N_HEADS, batch_size=BS)
model_fm.load_state_dict(torch.load("models/FMNIST/EBM_10_epochs_(1, 28, 28)_shape_3_heads.pth", map_location=DEVICE, weights_only=False))

model_olivetti = EBM(image_shape=IMG_SHAPE_Olivetti, n_heads=N_HEADS, batch_size=BS)
model_olivetti.load_state_dict(torch.load("models/OLIVETTI/EBM_10_epochs_(1, 64, 64)_shape_3_heads_1782916208844051426.pth", map_location=DEVICE, weights_only=False))

sampler_fm = ReplaySampler(model=model_fm, img_shape=IMG_SHAPE, buffer_size=BUFFER_SIZE, noise_fraction=NOISE_FRACTION, device=DEVICE)
sampler_olivetti = ReplaySampler(model=model_olivetti, img_shape=IMG_SHAPE_Olivetti, buffer_size=BUFFER_SIZE, noise_fraction=NOISE_FRACTION, device=DEVICE)


In [20]:
ebm_fm_samp = sampler_fm.generate(100, steps = 100, batch_size=32)
ebm_oliv_samp = sampler_olivetti.generate(100, steps = 100, batch_size=32)

In [31]:
print("FASHION MNIST - EBM")
evaluate_sampling_fid(real_images=test_samp_fm, fake_images=ebm_fm_samp)

print("OLIVETTI - EBM")
evaluate_sampling_fid(real_images=test_samp_oliv, fake_images=ebm_oliv_samp)

FASHION MNIST - EBM
FID: 185.35047912597656
OLIVETTI - EBM
FID: 420.6712951660156
